In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [2]:
df = pd.read_csv('cold-start-consumption-forecasting-training-data.csv', sep=';')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df.sort_values('timestamp', inplace=True)
df

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
1221786,18469,2,2013-01-01 03:00:00+00:00,0.500000,1.000000,1.000000,52561.679251,51033.723459,49964.154405,53631.248306
952980,778656,44,2013-01-01 03:00:00+00:00,-1.300000,-1.020000,-0.540000,2859.612151,3356.880013,3050.939778,3347.313631
29192,18470,2,2013-01-01 04:00:00+00:00,0.500000,0.500000,1.000000,55159.204098,52561.679251,51033.723459,62340.596321
847716,778657,44,2013-01-01 04:00:00+00:00,-1.900000,-1.300000,-1.020000,3392.386005,2859.612151,3356.880013,3286.419935
772553,778658,44,2013-01-01 05:00:00+00:00,-2.066667,-1.900000,-1.300000,2998.876587,3392.386005,2859.612151,2996.117054
...,...,...,...,...,...,...,...,...,...,...
1161832,1137480,61,2017-12-10 21:00:00+00:00,11.150000,10.100000,9.983333,16629.595022,17860.342098,20625.682268,20886.853284
1015224,1137481,61,2017-12-10 22:00:00+00:00,11.483333,11.150000,10.100000,17715.247089,16629.595022,17860.342098,18691.651149
932456,1137482,61,2017-12-10 23:00:00+00:00,11.925000,11.483333,11.150000,17295.325063,17715.247089,16629.595022,19847.290220
85313,1137483,61,2017-12-11 00:00:00+00:00,12.550000,11.925000,11.483333,17930.329102,17295.325063,17715.247089,20741.758275


In [3]:
# Select features and target
features = df[['temperature', 'temperature_-1', 'temperature_-2', 'load', 'load_-1', 'load_-2']]
target = df['target']

# Normalize features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_features = scaler.fit_transform(features)

# Reshape for LSTM [samples, time steps, features]
X = scaled_features.reshape((scaled_features.shape[0], 1, scaled_features.shape[1]))
X

array([[[0.2674316 , 0.27625772, 0.27625772, 0.03185159, 0.03092562,
         0.03027744]],

       [[0.23565755, 0.24060018, 0.24907326, 0.00173135, 0.0020327 ,
         0.0018473 ]],

       [[0.2674316 , 0.2674316 , 0.27625772, 0.03342573, 0.03185159,
         0.03092562]],

       ...,

       [[0.46910856, 0.46131215, 0.45542807, 0.01047962, 0.0107341 ,
         0.01007618]],

       [[0.48014122, 0.46910856, 0.46131215, 0.01086444, 0.01047962,
         0.0107341 ]],

       [[0.47984701, 0.48014122, 0.46910856, 0.01092755, 0.01086444,
         0.01047962]]])

In [4]:
# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.2, random_state=42)

# Define the LSTM model
model = Sequential()
model.add(LSTM(units=10, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2))
model.add(LSTM(units=10, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=1)) # Output layer

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# Train the model
model.fit(X_train, y_train, epochs=3, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/3
35062/35062 [==============================] - 28s 763us/step - loss: 29117216768.0000 - val_loss: 29197359104.0000
Epoch 2/3
35062/35062 [==============================] - 27s 763us/step - loss: 29082611712.0000 - val_loss: 29162721280.0000
Epoch 3/3
35062/35062 [==============================] - 26s 748us/step - loss: 29048207360.0000 - val_loss: 29128239104.0000


In [7]:
# Make predictions
trainPredict = model.predict(trainX)
testPredict = model.predict(testX)

# Invert predictions
trainPredict = scaler.inverse_transform(trainPredict)
trainY = scaler.inverse_transform([trainY])
testPredict = scaler.inverse_transform(testPredict)
testY = scaler.inverse_transform([testY])

# Calculate root mean squared error
trainScore = np.sqrt(mean_squared_error(trainY[0], trainPredict[:,0]))
print('Train Score: %.2f RMSE' % (trainScore))
testScore = np.sqrt(mean_squared_error(testY[0], testPredict[:,0]))
print('Test Score: %.2f RMSE' % (testScore))

NameError: name 'trainX' is not defined

In [6]:
np.sqrt(loss)

170669.97130133936